# BirdCLEF 2026 — Baseline Training (Colab)

EfficientNet-B0 SED baseline with FocalBCE loss, MixUp augmentation, single fold.

**Runtime**: GPU (T4) — Go to Runtime > Change runtime type > T4 GPU

In [ ]:
# === 1. Setup ===
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_2d7b97262c459561596c3b49927c7d29'  # Replace with your token

!pip install -q torch torchaudio timm pytorch-lightning scikit-learn pandas numpy pyyaml kaggle
# nnAudio is optional, we fall back to torchaudio
!pip install -q nnAudio 2>/dev/null || echo 'nnAudio not available, using torchaudio fallback'

In [ ]:
# === 2. Download competition data ===
!kaggle competitions download birdclef-2026 -p /content/data
!cd /content/data && unzip -q -o birdclef-2026.zip -d birdclef-2026
!ls /content/data/birdclef-2026/

In [ ]:
# === 3. Clone project code from GitHub ===
!git clone https://github.com/birdclef-hive/task--birdclef-2026.git /content/project 2>/dev/null || echo 'Already cloned'
os.chdir('/content/project')
import sys
sys.path.insert(0, '/content/project')

In [ ]:
# === 4. Quick EDA ===
import pandas as pd
import numpy as np

DATA_DIR = '/content/data/birdclef-2026'
AUDIO_DIR = f'{DATA_DIR}/train_audio'

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
train_df['primary_label'] = train_df['primary_label'].astype(str)
taxonomy = pd.read_csv(f'{DATA_DIR}/taxonomy.csv')
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)

# Species list from taxonomy (234 species, includes those with 0 training samples)
species_list = sorted(taxonomy['primary_label'].unique().tolist())
num_classes = len(species_list)

print(f'Train samples: {len(train_df)}')
print(f'Species in taxonomy: {num_classes}')
print(f'Species with training data: {train_df["primary_label"].nunique()}')
print(f'Columns: {list(train_df.columns)}')
print(f'\nClass distribution:')
print(train_df['class_name'].value_counts())
print(f'\nCollection sources:')
print(train_df['collection'].value_counts())
print(f'\nSpecies distribution (top 10):')
print(train_df['primary_label'].value_counts().head(10))
print(f'\nSpecies distribution (bottom 10):')
print(train_df['primary_label'].value_counts().tail(10))

In [ ]:
# === 5. Check audio files ===
import torchaudio
from pathlib import Path

print(f'Audio dir: {AUDIO_DIR}')
subdirs = sorted(os.listdir(AUDIO_DIR))
print(f'Subdirectories: {len(subdirs)}')
print(f'First 5: {subdirs[:5]}')

# Verify filename format matches
sample_fname = train_df['filename'].iloc[0]
sample_path = os.path.join(AUDIO_DIR, sample_fname)
print(f'\nSample filename: {sample_fname}')
print(f'Full path: {sample_path}')
print(f'Exists: {os.path.exists(sample_path)}')

In [ ]:
# === 6. Load a sample audio and verify pipeline ===
from src.transforms import AudioTransform
from src.dataset import load_audio

sample_path = os.path.join(AUDIO_DIR, train_df['filename'].iloc[0])
print(f'Loading: {sample_path}')

waveform = load_audio(sample_path, target_sr=32000, max_duration=30)
print(f'Waveform shape: {waveform.shape}')
print(f'Duration: {waveform.shape[1] / 32000:.1f}s')

transform = AudioTransform(train=False)
mel = transform(waveform[:, :160000])  # First 5s
print(f'Mel shape: {mel.shape}')

In [ ]:
# === 7. Build config and verify model ===
import yaml
import torch
from src.models.sed_model import SEDModel
from src.models.losses import build_loss

with open('configs/base.yaml') as f:
    config = yaml.safe_load(f)

print(f'Number of classes: {num_classes}')

config['model'] = config.get('model', {})
config['model']['num_classes'] = num_classes
config['model']['backbone'] = 'tf_efficientnet_b0_ns'

# Verify model creation
model = SEDModel.from_config(config)
dummy = torch.randn(2, 1, 128, 312)
clip_pred, frame_pred = model(dummy)
print(f'clip_pred shape: {clip_pred.shape}')  # (2, num_classes)
print(f'frame_pred shape: {frame_pred.shape}')

# Verify loss
loss_fn = build_loss(config)
target = torch.zeros(2, num_classes)
target[0, 0] = 1.0
target[1, 5] = 1.0
loss = loss_fn(clip_pred, target)
print(f'Loss: {loss.item():.4f}')
print('\nModel and loss verified!')

In [ ]:
# === 8. Create folds and prepare dataloaders ===
from src.train import load_config, create_folds, BirdCLEFModule
from src.dataset import BirdCLEFDataset
from src.transforms import AudioTransform, AudioAugmentations
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# --- Config ---
config = load_config('configs/base.yaml')
config['model'] = config.get('model', {})
config['model']['num_classes'] = num_classes
config['model']['backbone'] = 'tf_efficientnet_b0_ns'

# Colab settings
config['training']['epochs'] = 15
config['training']['batch_size'] = 32
config['data']['num_workers'] = 2
config['training']['mixed_precision'] = True  # Use AMP on T4

# --- Create folds (grouped by author to prevent leakage) ---
metadata = create_folds(
    train_df, n_folds=5,
    stratify_by='primary_label',
    group_by='author',
)

print(f'Fold distribution:\n{metadata["fold"].value_counts().sort_index()}')

In [ ]:
# === 9. Train fold 0 ===
FOLD = 0

train_meta = metadata[metadata['fold'] != FOLD].reset_index(drop=True)
val_meta = metadata[metadata['fold'] == FOLD].reset_index(drop=True)
print(f'Train: {len(train_meta)}, Val: {len(val_meta)}')

data_cfg = config.get('data', {})
train_transform = AudioTransform(data_cfg, train=True)
val_transform = AudioTransform(data_cfg, train=False)
augmentations = AudioAugmentations(config.get('augmentations', {}))

train_ds = BirdCLEFDataset(
    train_meta, AUDIO_DIR, species_list, data_cfg,
    train=True, transform=train_transform, augmentations=augmentations,
)
val_ds = BirdCLEFDataset(
    val_meta, AUDIO_DIR, species_list, data_cfg,
    train=False, transform=val_transform,
)

batch_size = config['training']['batch_size']
num_workers = config['data']['num_workers']

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=True,
)

# Test a batch
batch = next(iter(train_loader))
print(f'Batch mel shape: {batch["mel"].shape}')
print(f'Batch target shape: {batch["target"].shape}')
print(f'Target sum per sample (first 5): {batch["target"].sum(dim=1)[:5].tolist()}')

In [ ]:
# === 10. Train! ===
module = BirdCLEFModule(config)

OUTPUT_DIR = '/content/models/fold_0'
os.makedirs(OUTPUT_DIR, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        dirpath=OUTPUT_DIR,
        filename='best-{epoch:02d}-{val/macro_auc:.4f}',
        monitor='val/macro_auc',
        mode='max',
        save_top_k=1,
        save_last=True,
    ),
    EarlyStopping(
        monitor='val/macro_auc',
        mode='max',
        patience=5,
        verbose=True,
    ),
]

trainer = pl.Trainer(
    max_epochs=config['training']['epochs'],
    callbacks=callbacks,
    default_root_dir=OUTPUT_DIR,
    precision='16-mixed',
    accelerator='gpu',
    devices=1,
    log_every_n_steps=10,
)

trainer.fit(module, train_loader, val_loader)

In [ ]:
# === 11. Results ===
print(f'\nBest checkpoint: {callbacks[0].best_model_path}')
print(f'Best val/macro_auc: {callbacks[0].best_model_score:.4f}')

# Save to Google Drive for persistence
try:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/birdclef-2026/models/
    !cp -r /content/models/fold_0/ /content/drive/MyDrive/birdclef-2026/models/fold_0/
    print('Checkpoint saved to Google Drive!')
except:
    print('Google Drive not available — download checkpoint manually')

In [ ]:
# === 12. Submit to Hive (optional) ===
# Run this if you have hive-evolve installed
# !pip install -q hive-evolve
# !HIVE_SERVER=http://YOUR_SERVER:8000 hive run submit \
#     -m 'Baseline: EfficientNet-B0 SED, fold 0, FocalBCE, 15 epochs' \
#     --score {callbacks[0].best_model_score:.4f} \
#     --parent none \
#     --tldr 'EfficientNet-B0 baseline'
print('Training complete! Next steps:')
print('1. If score > 0.85, start pseudo-labeling')
print('2. Try EfficientNetV2-S or ECA-NFNet-L0 backbone')
print('3. Add more folds and ensemble')